# 03 - Review unmatched PC names

This notebook prepares manual review candidates for PC researchers whose
original name and mapped name do not appear exactly in the exploded
cited author table.

## 1. Setup

In [1]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists() and (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError(
        "Could not find the repository root. Launch Jupyter from the repo root "
        "or set PYTHONPATH to the folder containing project_setup.py."
    )

import os
import sys
from difflib import SequenceMatcher
from pathlib import Path

os.environ.setdefault("ARROW_USER_SIMD_LEVEL", "NONE")

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import pandas as pd
from IPython.display import display

from author_matching import normalize_name, short_openalex_id
from identity_validation import short_orcid
from project_setup import ensure_dirs, setup_project

setup = setup_project()
PROJECT = setup.project_folder

STEP_2_PREPARED = PROJECT / "step_2_data" / "prepared" / "all_papers"
STEP_3_SUMMARY = PROJECT / "step_3_artifacts" / "summary_tables"
STEP_3_CHECKS = PROJECT / "step_3_artifacts" / "check_tables"
MANUAL_UPDATE_DIR = STEP_3_CHECKS / "manual_update"

ensure_dirs(STEP_3_CHECKS, MANUAL_UPDATE_DIR)

IDENTITY_TABLE_PATH = STEP_3_SUMMARY / "pc_researcher_identity_validation.csv"
REF_AUTHORS_PATH = STEP_2_PREPARED / "all_ref_authors_exploded.parquet"

CANDIDATE_TABLE_OUT = STEP_3_CHECKS / "unmatched_pc_name_close_candidates.csv"
CANDIDATE_SUMMARY_OUT = STEP_3_SUMMARY / "close_name_summary.csv"
MANUAL_SELECTION_CSV = MANUAL_UPDATE_DIR / "manual_selection.csv"
MANUAL_DECISION_CSV = MANUAL_UPDATE_DIR / "unmatched_pc_name_close_candidates.csv"
MANUAL_DECISION_NUMBERS = MANUAL_UPDATE_DIR / "unmatched_pc_name_close_candidates.numbers"
MANUAL_CITATIONS_OUT = STEP_3_CHECKS / "unmatched_pc_cites.csv"
MANUAL_CITATION_SUMMARY_OUT = STEP_3_SUMMARY / "manual_unmatched_pc_citation_summary.csv"
MANUAL_DUPLICATE_TARGETS_OUT = STEP_3_CHECKS / "manual_accepted_duplicate_targets.csv"

print(f"Project folder: {PROJECT}")
print(f"Run mode: {setup.run_mode}")
print(f"Overwrite artifacts: {setup.overwrite_artifacts}")

Project folder: /Users/endersari/2026-02-citations-vs-pc-memberships
Run mode: fast
Overwrite artifacts: True


## 2. Read unmatched PC researchers and cited author names

In [2]:
identity = pd.read_csv(IDENTITY_TABLE_PATH)
ref_authors = pd.read_parquet(
    REF_AUTHORS_PATH,
    columns=[
        "issue",
        "work_id",
        "referenced_work_id",
        "ref_author_name",
        "ref_author_id",
        "ref_orcid",
    ],
)

unmatched = identity[
    identity["exploded_name_match_status"].eq("no_match_with_exploded_data")
].copy()

print(f"PC researchers in identity table: {len(identity):,}")
print(f"Researchers without original-or-mapped exact name match: {len(unmatched):,}")
display(
    unmatched[
        [
            "canonical_researchr_id",
            "name",
            "mapped_name_for_matching",
            "identity_layer",
            "identity_layer_reason",
        ]
    ].head(10)
)

PC researchers in identity table: 952
Researchers without original-or-mapped exact name match: 61


,canonical_researchr_id,name,mapped_name_for_matching,identity_layer,identity_layer_reason
22,alexkavvos,Alex Kavvos,Alex Kavvos,Layer 2,name_not_found_in_referenced_authors
35,alexeyloginov,Alexey Loginov,Alexey Loginov,Layer 2,name_not_found_in_referenced_authors
39,amirkafshdargoharshady1,Amir K. Goharshady,Amir K. Goharshady,Layer 2,name_not_found_in_referenced_authors
68,andrzejmurawski,Andrzej Murawski,Andrzej Murawski,Layer 2,name_not_found_in_referenced_authors
74,angelicamoreira,Angelica Moreira,Angélica Aparecida Moreira,Layer 2,name_not_found_in_referenced_authors
80,anthonywidjajalin,Anthony Widjaja Lin,Anthony Widjaja Lin,Layer 2,name_not_found_in_referenced_authors
84,arielkellison,Ariel E. Kellison,Ariel E. Kellison,Layer 2,name_not_found_in_referenced_authors
132,brentyorgey,Brent Yorgey,Brent Yorgey,Layer 2,name_not_found_in_referenced_authors
148,casperbach,Casper Bach,Casper Bach,Layer 2,name_not_found_in_referenced_authors
180,colingordon,Colin Gordon,Colin Gordon,Layer 2,name_not_found_in_referenced_authors


## 3. Build a cited author name index

I collapse the exploded cited author data to one row per normalized
cited author name. The evidence columns help during manual review.

In [3]:
def unique_list(values, limit=None):
    clean = sorted({
        str(value)
        for value in values
        if pd.notna(value) and str(value).strip()
    })
    if limit is None:
        return clean
    return clean[:limit]


def most_common_value(values):
    clean = [str(value) for value in values if pd.notna(value) and str(value).strip()]
    if not clean:
        return None
    return pd.Series(clean).value_counts().index[0]


ref = ref_authors.copy()
ref["ref_author_id"] = ref["ref_author_id"].map(short_openalex_id)
ref["ref_orcid"] = ref["ref_orcid"].map(short_orcid)
ref["ref_norm"] = ref["ref_author_name"].map(normalize_name)
ref = ref[ref["ref_norm"].ne("")].copy()

ref_index = (
    ref.groupby("ref_norm", dropna=False)
    .agg(
        primary_ref_author_name=("ref_author_name", most_common_value),
        ref_author_name_variants=("ref_author_name", lambda x: unique_list(x, limit=8)),
        n_evidence_rows=("ref_author_name", "size"),
        n_citing_papers=("work_id", "nunique"),
        n_cited_works=("referenced_work_id", "nunique"),
        n_openalex_author_ids=("ref_author_id", "nunique"),
        openalex_author_ids=("ref_author_id", lambda x: unique_list(x, limit=12)),
        openalex_orcids=("ref_orcid", lambda x: unique_list(x, limit=12)),
        cited_in_issues=("issue", lambda x: unique_list(x, limit=12)),
    )
    .reset_index()
)

print(f"Unique normalized cited-author names: {len(ref_index):,}")
display(ref_index.head())

Unique normalized cited-author names: 54,795


,ref_norm,primary_ref_author_name,ref_author_name_variants,n_evidence_rows,n_citing_papers,n_cited_works,n_openalex_author_ids,openalex_author_ids,openalex_orcids,cited_in_issues
0,& micha,& Micha,[& Micha],3,3,1,0,[],[],"[ICFP, POPL]"
1,& morris,& Morris,[& Morris],3,3,1,0,[],[],"[ICFP, POPL]"
2,( map,( Map,[( Map],1,1,1,0,[],[],[PLDI]
3,( ozer,( Ozer,[( Ozer],1,1,1,0,[],[],[PLDI]
4,(postech),(postech),[(postech)],1,1,1,0,[],[],[PLDI]


## 4. Generate close name candidates

The scoring is only a sorting aid. A high score does not automatically
mean the candidate is correct.

In [4]:
def name_tokens(text):
    return normalize_name(text).split()


def ratio(left, right):
    if not left or not right:
        return 0.0
    return 100.0 * SequenceMatcher(None, left, right).ratio()


def token_sort_text(text):
    return " ".join(sorted(name_tokens(text)))


def token_overlap_score(left, right):
    left_tokens = set(name_tokens(left))
    right_tokens = set(name_tokens(right))
    if not left_tokens or not right_tokens:
        return 0.0
    return 100.0 * len(left_tokens & right_tokens) / max(len(left_tokens), len(right_tokens))


def surname(text):
    tokens = name_tokens(text)
    return tokens[-1] if tokens else ""


def first_initial(text):
    tokens = name_tokens(text)
    return tokens[0][0] if tokens and tokens[0] else ""


token_to_ref_rows = {}
surname_to_ref_rows = {}
for ref_row_index, candidate in ref_index.iterrows():
    candidate_norm = candidate["ref_norm"]
    for token in set(name_tokens(candidate_norm)):
        token_to_ref_rows.setdefault(token, set()).add(ref_row_index)
    candidate_surname = surname(candidate_norm)
    if candidate_surname:
        surname_to_ref_rows.setdefault(candidate_surname, set()).add(ref_row_index)


def candidate_pool_indices(query_norm):
    pool = set()
    for token in set(name_tokens(query_norm)):
        pool.update(token_to_ref_rows.get(token, set()))

    query_surname = surname(query_norm)
    if query_surname:
        pool.update(surname_to_ref_rows.get(query_surname, set()))

    return sorted(pool)


def score_pair(query_norm, candidate_norm):
    full_ratio = ratio(query_norm, candidate_norm)
    token_sort_ratio = ratio(token_sort_text(query_norm), token_sort_text(candidate_norm))
    overlap = token_overlap_score(query_norm, candidate_norm)
    same_surname = surname(query_norm) == surname(candidate_norm)
    same_initial = first_initial(query_norm) == first_initial(candidate_norm)

    score = max(full_ratio, token_sort_ratio)
    if same_surname:
        score += 6
    if same_surname and same_initial:
        score += 4
    if overlap >= 50:
        score += 4

    return {
        "score": min(score, 100.0),
        "full_ratio": full_ratio,
        "token_sort_ratio": token_sort_ratio,
        "token_overlap": overlap,
        "same_surname": same_surname,
        "same_first_initial": same_initial,
    }


def review_bucket(row):
    strong_name_evidence = (
        row["token_overlap"] >= 60
        or row["full_ratio"] >= 92
        or row["token_sort_ratio"] >= 92
    )
    if row["candidate_score"] >= 92 and row["same_surname"] and strong_name_evidence:
        return "strong_candidate"
    if row["candidate_score"] >= 84 and (
        row["token_overlap"] >= 40
        or row["full_ratio"] >= 82
        or row["token_sort_ratio"] >= 82
    ):
        return "possible_candidate"
    if row["same_surname"] and row["same_first_initial"] and row["token_overlap"] >= 40:
        return "possible_candidate"
    return "weak_candidate"


candidate_rows = []
top_k = 12

for _, person in unmatched.iterrows():
    query_names = [
        ("original_name", person["name"]),
        ("mapped_name", person["mapped_name_for_matching"]),
    ]
    seen = set()
    person_candidates = []

    for query_source, query_name in query_names:
        query_norm = normalize_name(query_name)
        if not query_norm:
            continue

        for ref_row_index in candidate_pool_indices(query_norm):
            candidate = ref_index.loc[ref_row_index]
            candidate_norm = candidate["ref_norm"]
            scores = score_pair(query_norm, candidate_norm)

            if scores["score"] < 72:
                continue

            key = (candidate_norm, query_source)
            if key in seen:
                continue
            seen.add(key)

            person_candidates.append(
                {
                    "canonical_researchr_id": person["canonical_researchr_id"],
                    "pc_name": person["name"],
                    "mapped_name_for_matching": person["mapped_name_for_matching"],
                    "query_source": query_source,
                    "query_norm": query_norm,
                    "candidate_ref_author_name": candidate["primary_ref_author_name"],
                    "candidate_ref_norm": candidate_norm,
                    "candidate_score": round(scores["score"], 1),
                    "full_ratio": round(scores["full_ratio"], 1),
                    "token_sort_ratio": round(scores["token_sort_ratio"], 1),
                    "token_overlap": round(scores["token_overlap"], 1),
                    "same_surname": scores["same_surname"],
                    "same_first_initial": scores["same_first_initial"],
                    "n_evidence_rows": int(candidate["n_evidence_rows"]),
                    "n_citing_papers": int(candidate["n_citing_papers"]),
                    "n_cited_works": int(candidate["n_cited_works"]),
                    "n_openalex_author_ids": int(candidate["n_openalex_author_ids"]),
                    "openalex_author_ids": candidate["openalex_author_ids"],
                    "openalex_orcids": candidate["openalex_orcids"],
                    "ref_author_name_variants": candidate["ref_author_name_variants"],
                    "cited_in_issues": candidate["cited_in_issues"],
                }
            )

    person_candidates = sorted(
        person_candidates,
        key=lambda row: (
            row["candidate_score"],
            row["same_surname"],
            row["same_first_initial"],
            row["n_evidence_rows"],
        ),
        reverse=True,
    )[:top_k]

    for rank, row in enumerate(person_candidates, start=1):
        row["candidate_rank"] = rank
        candidate_rows.append(row)

candidates = pd.DataFrame(candidate_rows)

if len(candidates):
    candidates["review_bucket"] = candidates.apply(review_bucket, axis=1)
    candidates["manual_decision"] = ""
    candidates["manual_notes"] = ""

    ordered_columns = [
        "canonical_researchr_id",
        "pc_name",
        "mapped_name_for_matching",
        "query_source",
        "candidate_rank",
        "candidate_ref_author_name",
        "candidate_ref_norm",
        "candidate_score",
        "review_bucket",
        "full_ratio",
        "token_sort_ratio",
        "token_overlap",
        "same_surname",
        "same_first_initial",
        "n_evidence_rows",
        "n_citing_papers",
        "n_cited_works",
        "n_openalex_author_ids",
        "openalex_author_ids",
        "openalex_orcids",
        "ref_author_name_variants",
        "cited_in_issues",
        "manual_decision",
        "manual_notes",
    ]
    candidates = candidates[ordered_columns]

print(f"Candidate rows: {len(candidates):,}")
display(candidates.head(20))

Candidate rows: 510


,canonical_researchr_id,pc_name,mapped_name_for_matching,query_source,candidate_rank,candidate_ref_author_name,candidate_ref_norm,candidate_score,review_bucket,full_ratio,...,n_evidence_rows,n_citing_papers,n_cited_works,n_openalex_author_ids,openalex_author_ids,openalex_orcids,ref_author_name_variants,cited_in_issues,manual_decision,manual_notes
0,alexkavvos,Alex Kavvos,Alex Kavvos,original_name,1,G. A. Kavvos,g a kavvos,82.2,weak_candidate,76.2,...,61,37,12,1,[A5037972894],[0000-0001-7953-7975],[G. A. Kavvos],"[ICFP, OOPSLA, OOPSLA1, OOPSLA2, PLDI, POPL]",,
1,alexkavvos,Alex Kavvos,Alex Kavvos,mapped_name,2,G. A. Kavvos,g a kavvos,82.2,weak_candidate,76.2,...,61,37,12,1,[A5037972894],[0000-0001-7953-7975],[G. A. Kavvos],"[ICFP, OOPSLA, OOPSLA1, OOPSLA2, PLDI, POPL]",,
2,alexkavvos,Alex Kavvos,Alex Kavvos,original_name,3,Alex Graves,alex graves,76.7,weak_candidate,72.7,...,9,6,6,1,[A5043473089],[],[Alex Graves],"[OOPSLA, PLDI]",,
3,alexkavvos,Alex Kavvos,Alex Kavvos,mapped_name,4,Alex Graves,alex graves,76.7,weak_candidate,72.7,...,9,6,6,1,[A5043473089],[],[Alex Graves],"[OOPSLA, PLDI]",,
4,alexkavvos,Alex Kavvos,Alex Kavvos,original_name,5,Alex Davies,alex davies,76.7,weak_candidate,72.7,...,3,3,2,1,[A5101491074],[0000-0003-4917-5234],[Alex Davies],"[OOPSLA1, OOPSLA2, PLDI]",,
5,alexkavvos,Alex Kavvos,Alex Kavvos,mapped_name,6,Alex Davies,alex davies,76.7,weak_candidate,72.7,...,3,3,2,1,[A5101491074],[0000-0003-4917-5234],[Alex Davies],"[OOPSLA1, OOPSLA2, PLDI]",,
6,alexkavvos,Alex Kavvos,Alex Kavvos,original_name,7,Alex Knaust,alex knaust,76.7,weak_candidate,72.7,...,1,1,1,1,[A5052910558],[],[Alex Knaust],[OOPSLA],,
7,alexkavvos,Alex Kavvos,Alex Kavvos,mapped_name,8,Alex Knaust,alex knaust,76.7,weak_candidate,72.7,...,1,1,1,1,[A5052910558],[],[Alex Knaust],[OOPSLA],,
8,alexkavvos,Alex Kavvos,Alex Kavvos,original_name,9,Alex Galakatos,alex galakatos,76.0,weak_candidate,72.0,...,3,3,2,1,[A5039470845],[],[Alex Galakatos],"[OOPSLA, OOPSLA1, POPL]",,
9,alexkavvos,Alex Kavvos,Alex Kavvos,mapped_name,10,Alex Galakatos,alex galakatos,76.0,weak_candidate,72.0,...,3,3,2,1,[A5039470845],[],[Alex Galakatos],"[OOPSLA, OOPSLA1, POPL]",,


## 5. Summarize manual review coverage

In [5]:
if len(candidates):
    best_candidates = (
        candidates.sort_values(
            [
                "canonical_researchr_id",
                "candidate_score",
                "n_evidence_rows",
            ],
            ascending=[True, False, False],
        )
        .groupby("canonical_researchr_id", as_index=False)
        .head(1)
        .copy()
    )

    summary = pd.DataFrame(
        [
            {
                "description": "unmatched researchers reviewed",
                "n_researchers": len(unmatched),
                "share_of_unmatched_researchers": 1.0,
            },
            {
                "description": "researchers with at least one candidate",
                "n_researchers": best_candidates["canonical_researchr_id"].nunique(),
                "share_of_unmatched_researchers": best_candidates["canonical_researchr_id"].nunique()
                / len(unmatched),
            },
            {
                "description": "best candidate is strong",
                "n_researchers": int(best_candidates["review_bucket"].eq("strong_candidate").sum()),
                "share_of_unmatched_researchers": float(best_candidates["review_bucket"].eq("strong_candidate").mean()),
            },
            {
                "description": "best candidate is possible",
                "n_researchers": int(best_candidates["review_bucket"].eq("possible_candidate").sum()),
                "share_of_unmatched_researchers": float(best_candidates["review_bucket"].eq("possible_candidate").mean()),
            },
            {
                "description": "best candidate is weak",
                "n_researchers": int(best_candidates["review_bucket"].eq("weak_candidate").sum()),
                "share_of_unmatched_researchers": float(best_candidates["review_bucket"].eq("weak_candidate").mean()),
            },
        ]
    )
else:
    best_candidates = pd.DataFrame()
    summary = pd.DataFrame(
        [
            {
                "description": "unmatched researchers reviewed",
                "n_researchers": len(unmatched),
                "share_of_unmatched_researchers": 1.0,
            }
        ]
    )

display(summary)
display(
    best_candidates[
        [
            "pc_name",
            "mapped_name_for_matching",
            "candidate_ref_author_name",
            "candidate_score",
            "review_bucket",
            "n_evidence_rows",
            "openalex_author_ids",
            "openalex_orcids",
        ]
    ].head(25)
)

,description,n_researchers,share_of_unmatched_researchers
0,unmatched researchers reviewed,61,1.000000
1,researchers with at least one candidate,58,0.950820
2,best candidate is strong,27,0.465517
3,best candidate is possible,27,0.465517
4,best candidate is weak,4,0.068966


,pc_name,mapped_name_for_matching,candidate_ref_author_name,candidate_score,review_bucket,n_evidence_rows,openalex_author_ids,openalex_orcids
12,Alexey Loginov,Alexey Loginov,Alexey Guseynov,79.9,weak_candidate,1,[A5069523512],[]
0,Alex Kavvos,Alex Kavvos,G. A. Kavvos,82.2,weak_candidate,61,[A5037972894],[0000-0001-7953-7975]
24,Amir K. Goharshady,Amir K. Goharshady,Amir Kafshdar Goharshady,96.9,strong_candidate,144,[A5005241421],[0000-0003-1702-6584]
30,Andrzej Murawski,Andrzej Murawski,Andrzej S. Murawski,100.0,strong_candidate,43,[A5000119795],[0000-0002-4725-410X]
42,Angelica Moreira,Angélica Aparecida Moreira,Nelma Moreira,92.8,possible_candidate,3,[A5050789330],[0000-0003-0861-0105]
46,Anthony Widjaja Lin,Anthony Widjaja Lin,"Lin, Anthony Widjaja",100.0,possible_candidate,2,[],[0000-0003-4715-5096]
56,Ariel E. Kellison,Ariel E. Kellison,Ariel Kellison,100.0,strong_candidate,9,[A5059845097],[0000-0003-3177-7958]
60,Brent Yorgey,Brent Yorgey,Brent A. Yorgey,100.0,strong_candidate,23,[A5058418924],[0009-0005-0135-6134]
64,Casper Bach,Casper Bach,"Bach, Casper",99.7,possible_candidate,2,[],[0000-0003-0622-7639]
76,Colin Gordon,Colin Gordon,Colin S. Gordon,100.0,strong_candidate,67,[A5072587371],[0000-0002-9012-4490]


## 6. Save review tables

In [6]:
should_write = setup.overwrite_artifacts or not CANDIDATE_TABLE_OUT.exists()

if should_write:
    candidates.to_csv(CANDIDATE_TABLE_OUT, index=False)
    summary.to_csv(CANDIDATE_SUMMARY_OUT, index=False)
    print(f"wrote candidate table: {CANDIDATE_TABLE_OUT}")
    print(f"wrote summary table: {CANDIDATE_SUMMARY_OUT}")
else:
    print("Existing review tables were left unchanged.")
    print(f"candidate table: {CANDIDATE_TABLE_OUT}")
    print(f"summary table: {CANDIDATE_SUMMARY_OUT}")

wrote candidate table: /Users/endersari/2026-02-citations-vs-pc-memberships/step_3_artifacts/check_tables/unmatched_pc_name_close_candidates.csv
wrote summary table: /Users/endersari/2026-02-citations-vs-pc-memberships/step_3_artifacts/summary_tables/close_name_summary.csv


## 7. Citation counts for manual decisions

After manually reviewing the candidate table, export the edited Numbers
sheet as CSV and save it here:

`step_3_artifacts/check_tables/manual_update/manual_selection.csv`

I treat `accept` and `acceptandmerge` as accepted manual matches. The
citation counts below are based on accepted cited author name candidates.

In [7]:
def clean_decision(value):
    if pd.isna(value):
        return ""
    return str(value).strip().lower().replace(" ", "")


def join_unique(values, limit=None):
    clean = sorted({
        str(value)
        for value in values
        if pd.notna(value) and str(value).strip()
    })
    if limit is not None:
        clean = clean[:limit]
    return "; ".join(clean)


manual_source_candidates = [
    MANUAL_SELECTION_CSV,
    MANUAL_DECISION_CSV,
]
manual_source = next(
    (path for path in manual_source_candidates if path.exists()),
    None,
)

if manual_source is not None:
    manual_input = pd.read_csv(manual_source)
else:
    manual_input = candidates.copy()
    print("No CSV export found for manual decisions.")
    if MANUAL_DECISION_NUMBERS.exists():
        print(f"Found Numbers file: {MANUAL_DECISION_NUMBERS}")
        print("Please export it as CSV to make the manual decisions reproducible.")

required_input_columns = {
    "canonical_researchr_id",
    "pc_name",
    "mapped_name_for_matching",
    "candidate_ref_author_name",
    "manual_decision",
}
missing_manual_columns = required_input_columns - set(manual_input.columns)
if missing_manual_columns:
    raise ValueError(
        f"Manual candidate table is missing columns: {sorted(missing_manual_columns)}"
    )

if "manual_notes" not in manual_input.columns:
    manual_input["manual_notes"] = ""

full_candidate_columns = [
    column
    for column in candidates.columns
    if column not in {"manual_decision", "manual_notes"}
]
key_columns = [
    "canonical_researchr_id",
    "pc_name",
    "mapped_name_for_matching",
    "query_source",
    "candidate_rank",
    "candidate_ref_author_name",
]
missing_key_columns = [
    column
    for column in key_columns
    if column not in manual_input.columns or column not in candidates.columns
]
if missing_key_columns:
    raise ValueError(
        f"Manual candidate table is missing key columns: {missing_key_columns}"
    )

manual_candidates = candidates[full_candidate_columns].merge(
    manual_input[key_columns + ["manual_decision", "manual_notes"]],
    on=key_columns,
    how="left",
    validate="one_to_one",
)

manual_candidates["manual_decision_clean"] = manual_candidates["manual_decision"].map(clean_decision)
accepted_decisions = {"accept", "acceptandmerge"}
accepted_candidates = manual_candidates[
    manual_candidates["manual_decision_clean"].isin(accepted_decisions)
].copy()

if len(accepted_candidates):
    accepted_names = (
        accepted_candidates.groupby(
            [
                "canonical_researchr_id",
                "pc_name",
                "mapped_name_for_matching",
                "candidate_ref_norm",
            ],
            dropna=False,
        )
        .agg(
            candidate_ref_author_name=(
                "candidate_ref_author_name",
                lambda x: join_unique(x),
            ),
            manual_decision_clean=(
                "manual_decision_clean",
                lambda x: join_unique(x),
            ),
        )
        .reset_index()
    )
else:
    accepted_names = pd.DataFrame(
        columns=[
            "canonical_researchr_id",
            "pc_name",
            "mapped_name_for_matching",
            "candidate_ref_norm",
            "candidate_ref_author_name",
            "manual_decision_clean",
        ]
    )

ref_for_counts = ref[
    [
        "issue",
        "work_id",
        "referenced_work_id",
        "ref_author_name",
        "ref_author_id",
        "ref_orcid",
        "ref_norm",
    ]
].copy()

if len(accepted_names):
    accepted_ref_rows = accepted_names.merge(
        ref_for_counts,
        left_on="candidate_ref_norm",
        right_on="ref_norm",
        how="left",
    )

    citation_counts = (
        accepted_ref_rows.groupby("canonical_researchr_id", dropna=False)
        .agg(
            accepted_citation_rows=("work_id", "size"),
            accepted_citing_papers=("work_id", "nunique"),
            accepted_cited_works=("referenced_work_id", "nunique"),
            accepted_candidate_names=("candidate_ref_author_name", lambda x: join_unique(x)),
            accepted_ref_author_names=("ref_author_name", lambda x: join_unique(x, limit=20)),
            accepted_openalex_author_ids=("ref_author_id", lambda x: join_unique(x)),
            accepted_orcids=("ref_orcid", lambda x: join_unique(x)),
            accepted_issues=("issue", lambda x: join_unique(x)),
            accepted_decisions=("manual_decision_clean", lambda x: join_unique(x)),
        )
        .reset_index()
    )
else:
    citation_counts = pd.DataFrame(
        columns=[
            "canonical_researchr_id",
            "accepted_citation_rows",
            "accepted_citing_papers",
            "accepted_cited_works",
            "accepted_candidate_names",
            "accepted_ref_author_names",
            "accepted_openalex_author_ids",
            "accepted_orcids",
            "accepted_issues",
            "accepted_decisions",
        ]
    )

decision_lists = (
    manual_candidates.groupby("canonical_researchr_id", dropna=False)
    .agg(
        manual_decisions=("manual_decision_clean", lambda x: join_unique(x)),
        n_accepted_candidate_rows=(
            "manual_decision_clean",
            lambda x: int(pd.Series(x).isin(accepted_decisions).sum()),
        ),
        n_acceptandmerge_rows=(
            "manual_decision_clean",
            lambda x: int(pd.Series(x).eq("acceptandmerge").sum()),
        ),
    )
    .reset_index()
)

best_for_context = (
    candidates.sort_values(
        [
            "canonical_researchr_id",
            "candidate_score",
            "n_evidence_rows",
        ],
        ascending=[True, False, False],
    )
    .groupby("canonical_researchr_id", as_index=False)
    .head(1)[
        [
            "canonical_researchr_id",
            "candidate_ref_author_name",
            "candidate_score",
            "review_bucket",
            "n_evidence_rows",
        ]
    ]
    .rename(
        columns={
            "candidate_ref_author_name": "best_candidate_name",
            "candidate_score": "best_candidate_score",
            "review_bucket": "best_candidate_bucket",
            "n_evidence_rows": "best_candidate_evidence_rows",
        }
    )
)

manual_researcher_summary = (
    unmatched[
        [
            "canonical_researchr_id",
            "name",
            "mapped_name_for_matching",
            "identity_layer",
            "identity_layer_reason",
        ]
    ]
    .merge(decision_lists, on="canonical_researchr_id", how="left")
    .merge(citation_counts, on="canonical_researchr_id", how="left")
    .merge(best_for_context, on="canonical_researchr_id", how="left")
)

count_columns = [
    "n_accepted_candidate_rows",
    "n_acceptandmerge_rows",
    "accepted_citation_rows",
    "accepted_citing_papers",
    "accepted_cited_works",
]
for column in count_columns:
    manual_researcher_summary[column] = (
        manual_researcher_summary[column].fillna(0).astype(int)
    )

text_columns = [
    "manual_decisions",
    "accepted_candidate_names",
    "accepted_ref_author_names",
    "accepted_openalex_author_ids",
    "accepted_orcids",
    "accepted_issues",
    "accepted_decisions",
]
for column in text_columns:
    manual_researcher_summary[column] = manual_researcher_summary[column].fillna("")

def manual_match_status(row):
    if row["n_accepted_candidate_rows"] > 0:
        return "matched_by_manual_review"
    if row["manual_decisions"]:
        return "not_matched_after_manual_review"
    return "needs_manual_decision"


manual_researcher_summary["manual_match_status"] = manual_researcher_summary.apply(
    manual_match_status,
    axis=1,
)

manual_citation_summary = (
    manual_researcher_summary.groupby("manual_match_status", dropna=False)
    .agg(
        n_researchers=("canonical_researchr_id", "nunique"),
        total_accepted_citation_rows=("accepted_citation_rows", "sum"),
        mean_accepted_citation_rows=("accepted_citation_rows", "mean"),
        median_accepted_citation_rows=("accepted_citation_rows", "median"),
        max_accepted_citation_rows=("accepted_citation_rows", "max"),
    )
    .reset_index()
)

if len(accepted_names):
    duplicate_targets = (
        accepted_names.groupby("candidate_ref_norm", dropna=False)
        .agg(
            n_pc_researchers=("canonical_researchr_id", "nunique"),
            pc_researchers=("pc_name", lambda x: join_unique(x)),
            mapped_names=("mapped_name_for_matching", lambda x: join_unique(x)),
            accepted_candidate_names=(
                "candidate_ref_author_name",
                lambda x: join_unique(x),
            ),
            accepted_decisions=(
                "manual_decision_clean",
                lambda x: join_unique(x),
            ),
        )
        .reset_index()
    )
    duplicate_targets = duplicate_targets[
        duplicate_targets["n_pc_researchers"] > 1
    ].copy()
else:
    duplicate_targets = pd.DataFrame(
        columns=[
            "candidate_ref_norm",
            "n_pc_researchers",
            "pc_researchers",
            "mapped_names",
            "accepted_candidate_names",
            "accepted_decisions",
        ]
    )

display(manual_citation_summary)
display(
    manual_researcher_summary[
        [
            "name",
            "mapped_name_for_matching",
            "manual_match_status",
            "manual_decisions",
            "accepted_candidate_names",
            "accepted_citation_rows",
            "accepted_citing_papers",
            "accepted_cited_works",
            "accepted_openalex_author_ids",
            "best_candidate_name",
            "best_candidate_score",
            "best_candidate_bucket",
        ]
    ].sort_values(
        ["manual_match_status", "accepted_citation_rows", "name"],
        ascending=[True, False, True],
    )
)

if len(duplicate_targets):
    print("Accepted OpenAlex-side names used for more than one PC researcher:")
    display(duplicate_targets)
else:
    print("No duplicate accepted OpenAlex-side targets across PC researchers.")

if manual_source is not None:
    should_write_manual_outputs = (
        setup.overwrite_artifacts
        or not MANUAL_CITATIONS_OUT.exists()
        or not MANUAL_CITATION_SUMMARY_OUT.exists()
        or not MANUAL_DUPLICATE_TARGETS_OUT.exists()
    )
    if should_write_manual_outputs:
        manual_researcher_summary.to_csv(MANUAL_CITATIONS_OUT, index=False)
        manual_citation_summary.to_csv(MANUAL_CITATION_SUMMARY_OUT, index=False)
        duplicate_targets.to_csv(MANUAL_DUPLICATE_TARGETS_OUT, index=False)
        print(f"read manual decisions from: {manual_source}")
        print(f"wrote citation table: {MANUAL_CITATIONS_OUT}")
        print(f"wrote citation summary: {MANUAL_CITATION_SUMMARY_OUT}")
        print(f"wrote duplicate-target table: {MANUAL_DUPLICATE_TARGETS_OUT}")
    else:
        print("Existing manual citation outputs were left unchanged.")
else:
    print("Manual citation outputs were not written because no CSV export was found.")

,manual_match_status,n_researchers,total_accepted_citation_rows,mean_accepted_citation_rows,median_accepted_citation_rows,max_accepted_citation_rows
0,matched_by_manual_review,51,4186,82.078431,61.0,654
1,needs_manual_decision,10,0,0.000000,0.0,0


,name,mapped_name_for_matching,manual_match_status,manual_decisions,accepted_candidate_names,accepted_citation_rows,accepted_citing_papers,accepted_cited_works,accepted_openalex_author_ids,best_candidate_name,best_candidate_score,best_candidate_bucket
50,Peter O'Hearn,Peter O'Hearn,matched_by_manual_review,acceptandmerge,"O'Hearn, Peter; Peter W. O’Hearn",654,258,70,A5068136684,Peter W. O’Hearn,100.0,strong_candidate
45,Matthew J. Parkinson,Matthew J. Parkinson,matched_by_manual_review,acceptandmerge,J Parkinson; Matthew J. Parkinson J; Matthew P...,271,132,50,A5001725251; A5033231992,Matthew Parkinson,100.0,strong_candidate
54,Sriram Rajamani,Sriram Rajamani,matched_by_manual_review,acceptandmerge,S Rajamani; Sriram K. Rajamani,231,155,56,A5076139746,Sriram K. Rajamani,100.0,strong_candidate
34,Jonathan Ragan-Kelley,Jonathan Ragan-Kelley,matched_by_manual_review,accept,Jonathan Ragan‐Kelley,191,93,36,A5023577472,Jonathan Ragan‐Kelley,99.2,possible_candidate
55,Steve Blackburn,Steve Blackburn,matched_by_manual_review,accept,Stephen M. Blackburn,177,102,48,A5021388582,Stephen M. Blackburn,92.4,possible_candidate
...,...,...,...,...,...,...,...,...,...,...,...,...
39,Kristóf Marussy,Kristóf Marussy,needs_manual_decision,,,0,0,0,,NaN,NaN,NaN
42,Marie Kerjean,Marie Kerjean,needs_manual_decision,,,0,0,0,,NaN,NaN,NaN
46,Neha Agarwal,Neha Agarwal,needs_manual_decision,,,0,0,0,,Nimisha Agarwal,95.5,possible_candidate
56,Suparna Bhattacharya,Suparna Bhattacharya,needs_manual_decision,,,0,0,0,,Arnab Bhattacharya,99.5,possible_candidate


No duplicate accepted OpenAlex-side targets across PC researchers.
read manual decisions from: /Users/endersari/2026-02-citations-vs-pc-memberships/step_3_artifacts/check_tables/manual_update/manual_selection.csv
wrote citation table: /Users/endersari/2026-02-citations-vs-pc-memberships/step_3_artifacts/check_tables/unmatched_pc_cites.csv
wrote citation summary: /Users/endersari/2026-02-citations-vs-pc-memberships/step_3_artifacts/summary_tables/manual_unmatched_pc_citation_summary.csv
wrote duplicate-target table: /Users/endersari/2026-02-citations-vs-pc-memberships/step_3_artifacts/check_tables/manual_accepted_duplicate_targets.csv


## 8. How I use this table

I review the best candidates manually. A candidate can be accepted only
after checking external evidence, such as the researcher homepage,
DBLP, ORCID, DOI examples, or the OpenAlex author page.